# CDT quickstart

From the repository root, set up the notebook environment and launch this notebook with:

```bash
just notebook-setup
just notebook
```

The `justfile` owns those recipes. Inspect it if you want to see exactly which `uv`, Jupyter, and notebook commands are being run.

This notebook is the beginner path for a small 1+1-dimensional Causal Dynamical Triangulations run. It uses the Rust `cdt` binary as the simulation engine, then loads the generated `trace.csv` and `summary.json` files for a first look at acceptance, action, lattice volume, and the measured volume profile.

The notebook dependency group in `pyproject.toml` provides JupyterLab, IPykernel, nbconvert, Polars, Plotly, and Matplotlib. If `uv` is not installed, install it from Astral's uv documentation before running the `just` commands. If this clone does not already have a `cdt` binary available, the first code cell can build `target/release/cdt` with Cargo.

The default run is deliberately tiny so it finishes quickly. Treat the plots as a smoke test and a map of the output format, not as production physics.

## 1. Find or build the binary

From a local repository clone, this cell first honors `CDT_BINARY` when a cluster module or packaged release provides the executable. Otherwise it prefers an installed `cdt` on your `PATH`, then falls back to `target/release/cdt`, and finally runs `cargo build --release` if needed. If Cargo is not installed, install Rust with rustup or provide a binary through `CDT_BINARY`.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from pathlib import Path

import polars as pl


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "Cargo.toml").exists() and (path / "src" / "main.rs").exists():
            return path
    message = "Run this notebook from the causal-triangulations repository clone."
    raise RuntimeError(message)


def run_command(command: list[str], *, cwd: Path, env: dict[str, str] | None = None, timeout: int = 300) -> subprocess.CompletedProcess[str]:
    merged_env = os.environ.copy()
    if env is not None:
        merged_env.update(env)

    print("$", " ".join(command))
    result = subprocess.run(  # noqa: S603 - CDT binary is invoked as an argv list with shell=False, timeout, and controlled cwd.
        command,
        cwd=cwd,
        env=merged_env,
        text=True,
        capture_output=True,
        timeout=timeout,
        check=False,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if result.returncode != 0:
        message = f"command failed with exit code {result.returncode}: {' '.join(command)}\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}"
        raise RuntimeError(message)
    return result


def cdt_binary_path(root: Path) -> str:
    configured = os.environ.get("CDT_BINARY")
    if configured:
        binary = Path(configured).expanduser()
        if not binary.is_file():
            raise FileNotFoundError(f"CDT_BINARY does not point to a file: {binary}")
        return str(binary)

    binary_name = "cdt.exe" if os.name == "nt" else "cdt"
    installed_binary = shutil.which(binary_name)
    if installed_binary is not None:
        return installed_binary

    local_binary = root / "target" / "release" / binary_name
    if not local_binary.exists():
        run_command(["cargo", "build", "--release"], cwd=root)
    return str(local_binary)


ROOT = find_repo_root(Path.cwd().resolve())
cdt_binary = cdt_binary_path(ROOT)

print(f"Repository: {ROOT}")
print(f"Using cdt binary: {cdt_binary}")

## 2. Run a tiny simulation

The defaults below construct an open-boundary 1+1 CDT strip with 4 vertices per spatial slice and 5 time slices, then run 100 Metropolis proposal steps. The seed makes this exact short trajectory reproducible for a fixed crate version.

In [ ]:
PARAMETERS = {
    "vertices_per_slice": 4,
    "timeslices": 5,
    "topology": "open-boundary",
    "steps": 100,
    "thermalization_steps": 10,
    "measurement_frequency": 10,
    "temperature": 1.0,
    "cosmological_constant": 0.46209812037329684,
    "seed": 105,
}

RUN_DIR = ROOT / "target" / "notebooks" / "quickstart"
TRACE_PATH = RUN_DIR / "trace.csv"
SUMMARY_PATH = RUN_DIR / "summary.json"
RUN_DIR.mkdir(parents=True, exist_ok=True)

command = [
    cdt_binary,
    "--dimension",
    "2",
    "--vertices-per-slice",
    str(PARAMETERS["vertices_per_slice"]),
    "--timeslices",
    str(PARAMETERS["timeslices"]),
    "--topology",
    str(PARAMETERS["topology"]),
    "--steps",
    str(PARAMETERS["steps"]),
    "--thermalization-steps",
    str(PARAMETERS["thermalization_steps"]),
    "--measurement-frequency",
    str(PARAMETERS["measurement_frequency"]),
    "--temperature",
    str(PARAMETERS["temperature"]),
    "--cosmological-constant",
    str(PARAMETERS["cosmological_constant"]),
    "--seed",
    str(PARAMETERS["seed"]),
    "--simulate",
    "--output-csv",
    str(TRACE_PATH),
    "--output-json",
    str(SUMMARY_PATH),
]

run_command(command, cwd=ROOT, env={"RUST_LOG": "info"})

## 3. Load the outputs

`trace.csv` has one row per completed Metropolis step. `summary.json` stores configuration, aggregate diagnostics, final triangulation counts, move/proposal statistics, and scheduled measurements. The trace is loaded with Polars so the same notebook scales from this tiny run to larger CSV/Parquet analysis workflows.

In [ ]:
trace = pl.read_csv(TRACE_PATH).with_columns(
    pl.col("accepted").cast(pl.Boolean),
    pl.col("proposed").cast(pl.Boolean),
)

with SUMMARY_PATH.open(encoding="utf-8") as file:
    summary = json.load(file)

trace = (
    trace.with_row_index("row_index")
    .with_columns((pl.col("accepted").cast(pl.Int64).cum_sum() / (pl.col("row_index") + 1)).alias("running_acceptance"))
    .drop("row_index")
)

trace_summary = trace.select(
    pl.len().alias("trace_rows"),
    pl.col("accepted").sum().alias("accepted_steps"),
    pl.col("proposed").sum().alias("concrete_proposals"),
    pl.col("action").mean().alias("mean_action"),
    pl.col("triangles").mean().alias("mean_triangles"),
)

print(trace_summary)
print(json.dumps(summary["aggregate"], indent=2))
print(json.dumps(summary["final_triangulation"], indent=2))

## 4. Plot first diagnostics

Acceptance shows how often proposed transitions changed the chain state. Action and volume are quick checks for drift in this unfixed-volume ensemble. The volume profile is the measured triangle count per time slice.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError as exc:
    message = "Install the notebook environment with `just notebook-setup`, then rerun this cell."
    raise ModuleNotFoundError(message) from exc

steps = trace["step"].to_list()
actions = trace["action"].to_list()
vertices = trace["vertices"].to_list()
edges = trace["edges"].to_list()
triangles = trace["triangles"].to_list()
running_acceptance = trace["running_acceptance"].to_list()
profile = summary["aggregate"]["average_volume_profile"]
fluctuations = summary["aggregate"]["volume_fluctuations"]

figure, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)

axes[0, 0].plot(steps, running_acceptance, color="tab:green")
axes[0, 0].set_title("Running acceptance rate")
axes[0, 0].set_xlabel("Metropolis step")
axes[0, 0].set_ylabel("Accepted / steps")
axes[0, 0].set_ylim(0.0, 1.0)

axes[0, 1].plot(steps, actions, color="tab:blue")
axes[0, 1].set_title("Action trace")
axes[0, 1].set_xlabel("Metropolis step")
axes[0, 1].set_ylabel("Regge action")

axes[1, 0].plot(steps, vertices, label="vertices")
axes[1, 0].plot(steps, edges, label="edges")
axes[1, 0].plot(steps, triangles, label="triangles")
axes[1, 0].set_title("Lattice size")
axes[1, 0].set_xlabel("Metropolis step")
axes[1, 0].set_ylabel("count")
axes[1, 0].legend()

slice_indices = list(range(len(profile)))
axes[1, 1].bar(slice_indices, profile, yerr=fluctuations if fluctuations else None, color="tab:purple", alpha=0.8)
axes[1, 1].set_title("Average measured volume profile")
axes[1, 1].set_xlabel("time slice")
axes[1, 1].set_ylabel("triangles")

plt.show()

## 5. Try small changes

Now rerun the simulation cell after changing one value in `PARAMETERS`:

- Change `cosmological_constant` to see how the unfixed-volume ensemble grows or shrinks.
- Increase `steps` after the tiny smoke test behaves sensibly.
- Change `seed` to sample a different random trajectory.
- Switch `topology` to `toroidal`; keep at least 3 vertices per slice and 3 time slices.

For scriptable command-line patterns, see `docs/cli-examples.md`. For Slurm or Open OnDemand workflows, see `docs/hpc.md`.